```shell
cd ~/code/dp23rss_fork/
export PYTHONPATH=~/code/dp23rss_fork/
conda activate robodiff
jupyter notebook --ip="0.0.0.0"
```

In [1]:
from diffusion_policy.dataset.libero_dataset import LiberoFTDataset
import mediapy
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from robokit.debug_utils.images import concatenate_rgb_images, plot_action_wrt_time
from robokit.debug_utils.io import dataloader_speed_test

/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/robodiff/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


In [2]:
dataset = LiberoFTDataset(
    dataset_root="/home/geyuan/code/LIBERO-FT/libero/datasets/",
    dataset_subname="libero_force",
    hdf5_fns=[
        "tmp_replayed_wrench.hdf5",
    ],
    horizon=32, pad_before=0, pad_after=7,
    shape_meta={
        "obs": {
            "image": {
                "shape": [3, 128, 128],
                "type": "rgb"
            },
            "gripper": {
                "shape": [3, 128, 128],
                "type": "rgb"
            },
            "joint_state": {
                "shape": [6],
                "type": "low_dim"
            },
            "force" : {
                "shape": [6],
                "type": "low_dim"
            }
        },
        "action": {
            "shape": [7,]
        }
    },
    transform_color_jitter=True,
    load_future_obs=False,
)

{
  "obs.wrenches": {
    "count": 11723,
    "mean": [1.843, 3.2048, 4.8427, -0.1094, 0.4145, -0.2352],
    "std": [13.7611, 16.7171, 23.3075, 3.2563, 3.3554, 1.518],
    "min": [-129.3165, -408.8553, -111.3397, -18.2925, -36.0573, -28.0831],
    "max": [358.5761, 251.0575, 444.5059, 209.1904, 139.1617, 27.7514],
    "p01": [-32.5228, -23.6917, -16.7526, -5.5354, -7.3933, -5.824],
    "p99": [49.6666, 52.7222, 91.8009, 3.223, 11.6949, 3.3101]
  },
  "obs.ee_states": {
    "count": 11723,
    "mean": [-0.0054, 0.0946, 1.0694, 2.7379, -2.1385, -0.786],
    "std": [0.0625, 0.0821, 0.0811, 0.284, 0.7517, 0.4193],
    "min": [-0.219, -0.0989, 0.9104, 1.785, -3.1398, -2.0896],
    "max": [0.135, 0.2382, 1.2396, 3.5025, 0.1154, 0.1965],
    "p01": [-0.1991, -0.0659, 0.9154, 2.1359, -2.9067, -1.8726],
    "p99": [0.0941, 0.2172, 1.1921, 3.3519, 0.0024, -0.0357]
  },
  "action.actions": {
    "count": 11723,
    "mean": [0.0083, -0.0014, -0.1489, 0.0243, 0.0099, -0.0561, -1.0],
    "std": [0.2

In [3]:
# dataloader_speed_test(dataset, num_workers=48)

In [4]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import os

from robokit.debug_utils.images import concatenate_rgb_images, plot_action_wrt_time, plot_force_sensor_wrt_time, save_frames_as_video

def chw_to_hwc_uint8(x_chw: torch.Tensor) -> np.ndarray:
    """
    x_chw: (3,H,W), value in [-1,1] or [0,1]
    return: (H,W,3) uint8 [0,255]
    """
    x = x_chw.detach().cpu().float()

    # if in [-1,1] -> [0,1]
    if x.min() < 0:
        x = (x + 1.0) / 2.0

    x = x.clamp(0, 1)
    x = (x * 255.0).byte()  # uint8
    return x.permute(1, 2, 0).numpy()  # HWC


# -------------------------
# batch_data is what DataLoader yields
# batch_data: Dict, keys=['obs','action']
# obs['image']: (B, n_obs=1, 3, H, W)
# obs['gripper']: (B, 1, 3, H, W)   <- 可选用这个当 eye_in_hand 或第二路相机
# obs['force']: (B, 1, 6)
# action: (B, chunk_size, 7)
# -------------------------

vis_dataloader = DataLoader(dataset, batch_size=64, num_workers=48, shuffle=False)
batch_data = next(iter(vis_dataloader))

b = 47  # choose which sample in batch

obs = batch_data["obs"]
action = batch_data["action"]  # (B, T, 7)

# 取当前帧图像（n_obs=1，所以取 [:,0]）
agent_img_chw = obs["image"][b, 0]    # (3,H,W)
eye_img_chw   = obs["gripper"][b, 0]  # (3,H,W) 这里用 gripper 图当第二路；如果你有别的key就替换

agent_rgb = chw_to_hwc_uint8(agent_img_chw)  # (H,W,3)
eye_rgb   = chw_to_hwc_uint8(eye_img_chw)    # (H,W,3)

# force / wrench：你这里只有当前帧(1,6)，但你要画 chunk_size 的曲线
# 最简单：把当前帧的 force_torque 重复 T 次（或者如果你有未来 force 序列就换成那个）
force_torque = obs["force"][b, 0].detach().cpu().numpy()  # (6,)
T = action.shape[1]

future_wrenches = np.repeat(force_torque[None, :], repeats=T, axis=0)  # (T,6)

# action
action_np = action[b].detach().cpu().numpy()  # (T,7)

# 构造每一帧的双相机拼图
vis_agent_images = []
for _ in range(T):
    camera_frame = concatenate_rgb_images(agent_rgb, eye_rgb, vertical=True, resize_ratio=2.0)
    vis_agent_images.append(camera_frame)

# 画曲线帧（要求你已有这两个函数，且返回长度为 T 的 frame list）
action_frames, *_ = plot_action_wrt_time(action_np, only_last_frame=True)              # len == T
force_frames, *_  = plot_force_sensor_wrt_time(future_wrenches, only_last_frame=True)  # len == T

# 合成：camera + action + force
vis_merged_frames = [
    concatenate_rgb_images(cam, action_frames[-1], resize_ratio=1.0)
    for cam in vis_agent_images
]
vis_merged_frames = [
    concatenate_rgb_images(m, force_frames[-1], resize_ratio=1.0)
    for m in vis_merged_frames
]

save_frames_as_video(
    vis_merged_frames,
    "/mnt/dongxu-fs1/data-hdd/geyuan/code/LIBERO-FT/debug/tmp_libero_for_dp.mp4",
    fps=10,
)


'/mnt/dongxu-fs1/data-hdd/geyuan/code/LIBERO-FT/debug/tmp_libero_for_dp.mp4'